# VIX Futures Roll Trading Strategy — Full Backtest

**Based on Buehler & Cusatis (2018)** — *Trading the VIX Futures Roll Using Exchange-Traded Funds*

**Structure:**
1. Load VIX futures data & download ETF prices
2. Replicate Buehler's OOS period (Oct 2011 – Dec 2016) with his published thresholds
3. Full-sample backtest with 80/20 train/test split and threshold re-optimisation
4. Portfolio analysis: SPX + Vol strategy blend with weight optimisation
5. Key stress events (Volmageddon, COVID) kept **out-of-sample**

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import os, glob, re, warnings
from datetime import datetime, timedelta

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

try:
    import yfinance as yf
except ImportError:
    !pip install yfinance
    import yfinance as yf

print('All imports OK')

## 2. Configuration

In [ ]:
VIX_FUTURES_DIR = 'data/vix_futures'

# Buehler published thresholds
BUEHLER_SHORT =  0.0197
BUEHLER_LONG  = -0.2400

# Buehler OOS period
BUEHLER_OOS_START = '2011-10-04'
BUEHLER_OOS_END   = '2016-12-30'

# Month codes for old CBOE format
MONTH_CODES = {
    'F':1,'G':2,'H':3,'J':4,'K':5,'M':6,
    'N':7,'Q':8,'U':9,'V':10,'X':11,'Z':12
}

print('Config loaded.')

## 3. Load VIX Futures Data

In [ ]:
def parse_expiration_from_filename(fname):
    base = os.path.basename(fname)
    m = re.match(r'VX_(\d{4}-\d{2}-\d{2})\.csv', base)
    if m:
        return pd.Timestamp(m.group(1))
    m = re.match(r'CFE_([A-Z])(\d{2})_VX\.csv', base)
    if m:
        month_code, yy = m.group(1), int(m.group(2))
        year = 2000 + yy
        month = MONTH_CODES.get(month_code)
        if month is None:
            return None
        first_day = pd.Timestamp(year, month, 1)
        wed_offset = (2 - first_day.weekday()) % 7
        third_wed = first_day + pd.Timedelta(days=wed_offset + 14)
        return third_wed
    return None


def load_single_contract(fpath):
    try:
        df = pd.read_csv(fpath)
    except Exception:
        return None
    df.columns = df.columns.str.strip()
    if df['Trade Date'].str.contains('/').any():
        df['Trade Date'] = pd.to_datetime(df['Trade Date'], format='%m/%d/%Y')
    else:
        df['Trade Date'] = pd.to_datetime(df['Trade Date'])
    df = df.rename(columns={'Trade Date': 'date'})
    expiry = parse_expiration_from_filename(fpath)
    if expiry is None:
        return None
    df['expiry'] = expiry
    
    # Use Settle, fall back to Close when Settle is 0 (early 2013 files)
    df['Settle'] = pd.to_numeric(df['Settle'], errors='coerce')
    df['Close']  = pd.to_numeric(df['Close'], errors='coerce')
    df['Settle'] = df['Settle'].where(df['Settle'] > 0, df['Close'])
    df = df[df['Settle'] > 0].copy()
    return df[['date', 'expiry', 'Settle', 'Total Volume', 'Open Interest']].copy()


all_files = sorted(glob.glob(os.path.join(VIX_FUTURES_DIR, '*.csv')))
print(f'Found {len(all_files)} VIX futures CSV files')

frames = []
for f in all_files:
    df = load_single_contract(f)
    if df is not None and len(df) > 0:
        frames.append(df)

futures_all = pd.concat(frames, ignore_index=True)
futures_all = futures_all.sort_values(['date', 'expiry']).reset_index(drop=True)
futures_all = futures_all[futures_all['date'] >= '2008-01-01'].reset_index(drop=True)

print(f'Total rows: {len(futures_all):,}')
print(f'Date range: {futures_all.date.min().date()} to {futures_all.date.max().date()}')
print(f'Unique contracts: {futures_all.expiry.nunique()}')

## 4. Build Daily Term Structure

In [ ]:
def build_term_structure(futures_all):
    df = futures_all[futures_all['expiry'] > futures_all['date']].copy()
    df['dte'] = (df['expiry'] - df['date']).dt.days
    records = []
    for date, group in df.groupby('date'):
        g = group.sort_values('expiry')
        if len(g) < 2:
            continue
        f1, f2 = g.iloc[0], g.iloc[1]
        records.append({
            'date': date,
            'F1_settle': f1['Settle'], 'F1_expiry': f1['expiry'], 'F1_dte': f1['dte'],
            'F2_settle': f2['Settle'], 'F2_expiry': f2['expiry'], 'F2_dte': f2['dte'],
        })
    ts = pd.DataFrame(records)
    ts['date'] = pd.to_datetime(ts['date'])
    ts = ts.set_index('date').sort_index()
    return ts

term_structure = build_term_structure(futures_all)
print(f'Term structure: {len(term_structure)} trading days')
print(f'Range: {term_structure.index.min().date()} to {term_structure.index.max().date()}')

## 5. Download VIX Spot & ETF Prices

In [ ]:
def dl(ticker, start='2006-01-01', end='2026-12-31'):
    tmp = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)
    tmp = tmp[['Close']]
    if isinstance(tmp.columns, pd.MultiIndex):
        tmp.columns = tmp.columns.get_level_values(0)
    tmp.index = pd.to_datetime(tmp.index)
    if hasattr(tmp.index, 'levels'):
        tmp.index = tmp.index.get_level_values(0)
    tmp.index.name = 'date'
    return tmp

vix  = dl('^VIX').rename(columns={'Close': 'VIX'})
svxy = dl('SVXY').rename(columns={'Close': 'SVXY'})
vixy = dl('VIXY').rename(columns={'Close': 'VIXY'})
spy  = dl('SPY').rename(columns={'Close': 'SPY'})

for name, df in [('VIX', vix), ('SVXY', svxy), ('VIXY', vixy), ('SPY', spy)]:
    print(f'{name:5s}: {len(df)} rows, {df.index.min().date()} to {df.index.max().date()}')

## 6. Merge All Data & Compute Daily Roll

In [ ]:
data = term_structure.join(vix, how='inner')
data = data.join(svxy, how='left').join(vixy, how='left').join(spy, how='left')

# Trading days to expiry
data['F1_tdte'] = data.apply(
    lambda r: max(np.busday_count(r.name.date(), r['F1_expiry'].date()), 1), axis=1
)

# Daily roll: (F1 - VIX) / trading days to settlement
# Positive in contango, negative in backwardation
data['daily_roll'] = (data['F1_settle'] - data['VIX']) / data['F1_tdte']

# ETF daily returns
data['SVXY_ret'] = data['SVXY'].pct_change()
data['VIXY_ret'] = data['VIXY'].pct_change()
data['SPY_ret']  = data['SPY'].pct_change()

# Futures return proxy (for futures-only strategy)
data['F1_ret'] = data['F1_settle'].pct_change()

print(f'Merged dataset: {len(data)} rows')
print(f'Range: {data.index.min().date()} to {data.index.max().date()}')
data[['VIX', 'F1_settle', 'daily_roll', 'SVXY', 'VIXY']].head()

## 7. Strategy Engine

**Buehler rules:**
- Daily roll ≥ short threshold → **sell vol** (buy SVXY / short futures)
- Daily roll ≤ long threshold → **buy vol** (buy VIXY / long futures)
- In between → **cash**

Signal based on previous day's close (next-day execution).

In [ ]:
def generate_signals(data, short_thresh, long_thresh):
    """Generate signals: 'SVXY', 'VIXY', or 'CASH'."""
    prev_roll = data['daily_roll'].shift(1)
    signal = pd.Series('CASH', index=data.index)
    signal[prev_roll >= short_thresh] = 'SVXY'
    signal[prev_roll <= long_thresh]  = 'VIXY'
    return signal


def backtest(data, signal, label='Strategy', start=None, end=None):
    """
    Backtest using ETF returns (primary) and futures returns (comparison).
    Returns (equity_df, stats_dict).
    """
    df = data.loc[start:end].copy() if start else data.copy()
    sig = signal.loc[df.index]
    df['signal'] = sig
    df = df.dropna(subset=['SVXY_ret', 'VIXY_ret'])
    if len(df) == 0:
        return pd.DataFrame(), {}
    
    # ETF strategy returns
    df['etf_ret'] = 0.0
    df.loc[df['signal'] == 'SVXY', 'etf_ret'] = df.loc[df['signal'] == 'SVXY', 'SVXY_ret']
    df.loc[df['signal'] == 'VIXY', 'etf_ret'] = df.loc[df['signal'] == 'VIXY', 'VIXY_ret']
    
    # Futures strategy returns: short futures when SVXY signal, long when VIXY
    df['fut_ret'] = 0.0
    df.loc[df['signal'] == 'SVXY', 'fut_ret'] = -df.loc[df['signal'] == 'SVXY', 'F1_ret']
    df.loc[df['signal'] == 'VIXY', 'fut_ret'] = df.loc[df['signal'] == 'VIXY', 'F1_ret']
    
    # Equity curves
    df['etf_equity'] = (1 + df['etf_ret']).cumprod()
    df['fut_equity'] = (1 + df['fut_ret']).cumprod()
    df['spy_equity'] = (1 + df['SPY_ret']).cumprod()
    
    # Stats
    def calc_stats(rets, equity, name):
        n = len(rets)
        yrs = n / 252
        terminal = equity.iloc[-1]
        cagr = terminal ** (1/yrs) - 1 if yrs > 0 else 0
        vol = rets.std() * np.sqrt(252)
        sharpe = (rets.mean() * 252) / vol if vol > 0 else 0
        semi = rets[rets < 0].std() * np.sqrt(252)
        sortino = (rets.mean() * 252) / semi if semi > 0 else 0
        dd = (equity / equity.cummax()) - 1
        return {
            'Terminal ($1)': f'{terminal:.2f}',
            'CAGR': f'{cagr:.1%}', 'Vol (ann.)': f'{vol:.1%}',
            'Sharpe': f'{sharpe:.2f}', 'Sortino': f'{sortino:.2f}',
            'Max DD': f'{dd.min():.1%}',
        }
    
    n_trades = (df['signal'] != df['signal'].shift(1)).sum()
    stats = calc_stats(df['etf_ret'], df['etf_equity'], label)
    stats['Trades'] = n_trades
    stats['Days Short'] = (df['signal'] == 'SVXY').sum()
    stats['Days Long'] = (df['signal'] == 'VIXY').sum()
    stats['Days Cash'] = (df['signal'] == 'CASH').sum()
    stats['Period'] = f"{df.index.min().date()} to {df.index.max().date()}"
    
    return df, stats

print('Engine ready.')

## 8. Buehler Replication (Oct 2011 – Dec 2016)

Using the **published thresholds** (+0.0197 / −0.240) on the same OOS window.

In [ ]:
sig_buehler = generate_signals(data, BUEHLER_SHORT, BUEHLER_LONG)
bt_b, stats_b = backtest(data, sig_buehler, 'Buehler ETF',
                          start=BUEHLER_OOS_START, end=BUEHLER_OOS_END)

# ── Exhibit 6 replica ──
fig, ax = plt.subplots(figsize=(14, 7))
ax.plot(bt_b.index, bt_b['etf_equity'], 'k-', lw=2, label='SVXY/VIXY Strategy')
ax.plot(bt_b.index, bt_b['fut_equity'], color='gray', lw=1.5, label='Futures Only')
ax.plot(bt_b.index, bt_b['spy_equity'], color='lightgray', lw=1.5, label='S&P 500')
ax.set_title('Buehler Replication — Trading Strategy Results\n'
             f'({BUEHLER_OOS_START} to {BUEHLER_OOS_END})', fontsize=14)
ax.set_ylabel('Value of $1.00 Investment')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d/%Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print('\n=== Buehler Replication Stats ===')
for k, v in stats_b.items():
    print(f'  {k:20s}: {v}')

## 9. Full-Sample 80/20 Train/Test Split

We use all available data where both SVXY and VIXY exist.
The split is chosen so that **Volmageddon (Feb 2018)** and **COVID (Mar 2020)**
are in the **test set** — because if a strategy dies there, it's not real.

In [ ]:
# Full period where both ETFs trade
etf_data = data.dropna(subset=['SVXY', 'VIXY']).copy()
print(f'Full ETF period: {etf_data.index.min().date()} to {etf_data.index.max().date()}')
print(f'Total days: {len(etf_data)}')

# 80/20 split
split_idx = int(len(etf_data) * 0.80)
train_end = etf_data.index[split_idx - 1]
test_start = etf_data.index[split_idx]

print(f'\nTrain: {etf_data.index.min().date()} to {train_end.date()} ({split_idx} days)')
print(f'Test:  {test_start.date()} to {etf_data.index.max().date()} ({len(etf_data) - split_idx} days)')

# Check key events are in test set
for evt, dt in [('Volmageddon', '2018-02-05'), ('COVID crash', '2020-03-16')]:
    in_test = pd.Timestamp(dt) >= test_start
    print(f'  {evt} ({dt}): {"✅ in test set" if in_test else "❌ in train set"}')

## 10. Threshold Optimisation on Training Data

Grid search over short and long thresholds, optimising for **Sharpe ratio**
on the training set only.

In [ ]:
short_grid = np.arange(0.005, 0.10, 0.005)
long_grid  = np.arange(-0.50, -0.02, 0.02)

train_start = etf_data.index.min().strftime('%Y-%m-%d')
train_end_str = train_end.strftime('%Y-%m-%d')

results = []
for st in short_grid:
    for lt in long_grid:
        sig = generate_signals(data, st, lt)
        bt_tmp, stats_tmp = backtest(data, sig, '', start=train_start, end=train_end_str)
        if len(bt_tmp) == 0:
            continue
        sharpe_val = float(stats_tmp['Sharpe'])
        terminal_val = bt_tmp['etf_equity'].iloc[-1]
        dd_val = (bt_tmp['etf_equity'] / bt_tmp['etf_equity'].cummax() - 1).min()
        results.append({
            'short_thresh': round(st, 4),
            'long_thresh': round(lt, 4),
            'sharpe': sharpe_val,
            'terminal': terminal_val,
            'max_dd': dd_val,
        })

res_df = pd.DataFrame(results)

# Best by Sharpe
best = res_df.loc[res_df['sharpe'].idxmax()]
OPT_SHORT = best['short_thresh']
OPT_LONG  = best['long_thresh']

print(f'=== Optimal Thresholds (Train Set, max Sharpe) ===')
print(f'  Short threshold: {OPT_SHORT:.4f}  (Buehler: {BUEHLER_SHORT})')
print(f'  Long threshold:  {OPT_LONG:.4f}  (Buehler: {BUEHLER_LONG})')
print(f'  Train Sharpe:    {best["sharpe"]:.3f}')
print(f'  Train Terminal:  ${best["terminal"]:.2f}')
print(f'  Train Max DD:    {best["max_dd"]:.1%}')

# Heatmap
pivot = res_df.pivot_table(values='sharpe', index='long_thresh', columns='short_thresh')
fig, ax = plt.subplots(figsize=(14, 6))
im = ax.pcolormesh(pivot.columns, pivot.index, pivot.values, cmap='RdYlGn', shading='auto')
ax.plot(BUEHLER_SHORT, BUEHLER_LONG, 'k*', markersize=15, label='Buehler published')
ax.plot(OPT_SHORT, OPT_LONG, 'bs', markersize=10, label='Optimised (train)')
ax.set_xlabel('Short Threshold (contango → sell vol)')
ax.set_ylabel('Long Threshold (backwardation → buy vol)')
ax.set_title(f'Sharpe Ratio — Threshold Sensitivity (Train: {train_start} to {train_end_str})')
ax.legend(fontsize=11)
plt.colorbar(im, label='Sharpe Ratio')
plt.tight_layout()
plt.show()

## 11. Out-of-Sample Test (with Volmageddon & COVID)

We run the **optimised thresholds** on the held-out test set and compare to
Buehler's original thresholds and S&P 500 buy-and-hold.

In [ ]:
test_start_str = test_start.strftime('%Y-%m-%d')
test_end_str = etf_data.index.max().strftime('%Y-%m-%d')

# Optimised
sig_opt = generate_signals(data, OPT_SHORT, OPT_LONG)
bt_opt, stats_opt = backtest(data, sig_opt, 'Optimised',
                              start=test_start_str, end=test_end_str)

# Buehler original on test period
sig_btest = generate_signals(data, BUEHLER_SHORT, BUEHLER_LONG)
bt_btest, stats_btest = backtest(data, sig_btest, 'Buehler Original',
                                  start=test_start_str, end=test_end_str)

# ── Chart ──
fig, ax = plt.subplots(figsize=(14, 7))
ax.plot(bt_opt.index, bt_opt['etf_equity'], 'b-', lw=2, label=f'Optimised (ST={OPT_SHORT:.3f}, LT={OPT_LONG:.2f})')
ax.plot(bt_btest.index, bt_btest['etf_equity'], 'k--', lw=1.5, label='Buehler Original Thresholds')
ax.plot(bt_opt.index, bt_opt['spy_equity'], color='lightgray', lw=1.5, label='S&P 500')

# Mark events
for dt, lbl in [('2018-02-05', 'Volmageddon'), ('2020-03-16', 'COVID')]:
    evt = pd.Timestamp(dt)
    if evt >= bt_opt.index.min() and evt <= bt_opt.index.max():
        ax.axvline(evt, color='red', ls=':', alpha=0.5)
        ax.text(evt, ax.get_ylim()[1]*0.85, f' {lbl}', fontsize=9, color='red')

ax.set_title(f'Out-of-Sample Test ({test_start_str} to {test_end_str})', fontsize=14)
ax.set_ylabel('Value of $1.00 Investment')
ax.set_yscale('log')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f'{y:.1f}'))
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

# Stats comparison
comp = pd.DataFrame({'Optimised': stats_opt, 'Buehler Original': stats_btest}).T
print('\n=== Out-of-Sample Results ===')
display(comp)

## 12. Full Period Equity Curve (Train + Test)

In [ ]:
# Run optimised strategy on full period
full_start = etf_data.index.min().strftime('%Y-%m-%d')
full_end = etf_data.index.max().strftime('%Y-%m-%d')

bt_full, stats_full = backtest(data, sig_opt, 'Full Period', start=full_start, end=full_end)

fig, axes = plt.subplots(2, 1, figsize=(14, 10), height_ratios=[3, 1])

# Equity
ax = axes[0]
ax.plot(bt_full.index, bt_full['etf_equity'], 'b-', lw=1.5, label='Vol Strategy (optimised)')
ax.plot(bt_full.index, bt_full['spy_equity'], color='gray', lw=1, label='S&P 500')
ax.axvline(train_end, color='green', ls='--', alpha=0.5, label='Train/Test split')
for dt, lbl in [('2018-02-05', 'Volmageddon'), ('2020-03-16', 'COVID')]:
    evt = pd.Timestamp(dt)
    if evt >= bt_full.index.min():
        ax.axvline(evt, color='red', ls=':', alpha=0.4)
        ax.text(evt, ax.get_ylim()[1]*0.7, f' {lbl}', fontsize=9, color='red')
ax.set_yscale('log')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f'{y:.0f}'))
ax.set_title('Full Period: Vol Strategy vs S&P 500', fontsize=14)
ax.set_ylabel('Value of $1 (log scale)')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, which='both')

# Drawdown
ax2 = axes[1]
eq = bt_full['etf_equity']
dd = (eq / eq.cummax()) - 1
ax2.fill_between(dd.index, dd.values, 0, color='blue', alpha=0.3)
ax2.plot(dd.index, dd.values, 'b-', lw=0.5)
ax2.axvline(train_end, color='green', ls='--', alpha=0.5)
ax2.set_ylabel('Drawdown')
ax2.set_title(f'Strategy Drawdown (Max: {dd.min():.1%})')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('\n=== Full Period Stats ===')
for k, v in stats_full.items():
    print(f'  {k:20s}: {v}')

## 13. Portfolio: S&P 500 + Vol Strategy

We blend the vol strategy with S&P 500 buy-and-hold at various weights and find
the allocation that **maximises the Sharpe ratio** and the one that **minimises max drawdown**.

The portfolio is rebalanced daily (constant-weight).

In [ ]:
# Use the FULL period for portfolio analysis
port_data = bt_full[['etf_ret', 'SPY_ret']].dropna().copy()

weights = np.arange(0.0, 1.01, 0.01)  # vol strategy weight
port_results = []

for w_vol in weights:
    w_spy = 1 - w_vol
    port_ret = w_vol * port_data['etf_ret'] + w_spy * port_data['SPY_ret']
    port_eq = (1 + port_ret).cumprod()
    
    n_yrs = len(port_ret) / 252
    terminal = port_eq.iloc[-1]
    cagr = terminal ** (1/n_yrs) - 1
    vol = port_ret.std() * np.sqrt(252)
    sharpe = (port_ret.mean() * 252) / vol if vol > 0 else 0
    dd = (port_eq / port_eq.cummax() - 1).min()
    semi = port_ret[port_ret < 0].std() * np.sqrt(252)
    sortino = (port_ret.mean() * 252) / semi if semi > 0 else 0
    
    port_results.append({
        'w_vol': w_vol, 'w_spy': w_spy,
        'cagr': cagr, 'vol': vol, 'sharpe': sharpe,
        'sortino': sortino, 'max_dd': dd, 'terminal': terminal,
    })

port_df = pd.DataFrame(port_results)

# Best Sharpe
best_sharpe = port_df.loc[port_df['sharpe'].idxmax()]
# Min drawdown
best_dd = port_df.loc[port_df['max_dd'].idxmax()]  # max because DD is negative

print('=== Optimal Portfolio Weights ===')
print(f'\nMax Sharpe:    {best_sharpe["w_vol"]:.0%} Vol + {best_sharpe["w_spy"]:.0%} SPY')
print(f'  Sharpe={best_sharpe["sharpe"]:.2f}, CAGR={best_sharpe["cagr"]:.1%}, '
      f'Max DD={best_sharpe["max_dd"]:.1%}')
print(f'\nMin Drawdown:  {best_dd["w_vol"]:.0%} Vol + {best_dd["w_spy"]:.0%} SPY')
print(f'  Sharpe={best_dd["sharpe"]:.2f}, CAGR={best_dd["cagr"]:.1%}, '
      f'Max DD={best_dd["max_dd"]:.1%}')

# ── Plot metrics vs weight ──
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for ax, metric, title, fmt in [
    (axes[0,0], 'sharpe', 'Sharpe Ratio', '{:.2f}'),
    (axes[0,1], 'max_dd', 'Max Drawdown', '{:.0%}'),
    (axes[1,0], 'cagr', 'CAGR', '{:.0%}'),
    (axes[1,1], 'sortino', 'Sortino Ratio', '{:.2f}'),
]:
    ax.plot(port_df['w_vol'] * 100, port_df[metric], 'b-', lw=2)
    ax.set_xlabel('Vol Strategy Weight (%)')
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    # Mark key points
    for pct, color, lbl in [(0.10, 'orange', '10%'), (0.20, 'red', '20%'),
                             (best_sharpe['w_vol'], 'green', 'Best Sharpe')]:
        row = port_df.iloc[(port_df['w_vol'] - pct).abs().idxmin()]
        ax.axvline(pct * 100, color=color, ls='--', alpha=0.5)
    if metric == 'max_dd':
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0%}'))
    elif metric == 'cagr':
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0%}'))

plt.suptitle('Portfolio Metrics vs Vol Strategy Allocation', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 14. Portfolio Equity Curves

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), height_ratios=[3, 1])

# Selected allocations
allocations = [
    (0.00, '100% SPY', 'lightgray'),
    (0.10, '90/10 SPY/Vol', 'green'),
    (0.20, '80/20 SPY/Vol', 'blue'),
    (best_sharpe['w_vol'], f'{best_sharpe["w_spy"]:.0%}/{best_sharpe["w_vol"]:.0%} SPY/Vol (max Sharpe)', 'red'),
    (1.00, '100% Vol Strategy', 'black'),
]

ax = axes[0]
ax2 = axes[1]

for w_vol, label, color in allocations:
    port_ret = w_vol * port_data['etf_ret'] + (1 - w_vol) * port_data['SPY_ret']
    port_eq = (1 + port_ret).cumprod()
    ax.plot(port_eq.index, port_eq, color=color, lw=1.5 if w_vol in [0,1] else 2, label=label)
    dd = (port_eq / port_eq.cummax()) - 1
    ax2.plot(dd.index, dd, color=color, lw=0.8, alpha=0.7)

ax.axvline(train_end, color='green', ls='--', alpha=0.3, label='Train/Test split')
ax.set_yscale('log')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f'{y:.0f}'))
ax.set_title('Portfolio Equity Curves: S&P 500 + Vol Strategy Blends', fontsize=14)
ax.set_ylabel('Value of $1 (log scale)')
ax.legend(fontsize=10, loc='upper left')
ax.grid(True, alpha=0.3, which='both')

ax2.axvline(train_end, color='green', ls='--', alpha=0.3)
ax2.set_ylabel('Drawdown')
ax2.set_title('Drawdowns by Allocation')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 15. Summary Table — Key Allocations

In [ ]:
summary_rows = []
for w_vol, label in [(0.0, '100% SPY'), (0.10, '90/10'), (0.20, '80/20'),
                      (best_sharpe['w_vol'], f'Max Sharpe ({best_sharpe["w_spy"]:.0%}/{best_sharpe["w_vol"]:.0%})'),
                      (1.0, '100% Vol Strategy')]:
    row = port_df.iloc[(port_df['w_vol'] - w_vol).abs().idxmin()]
    summary_rows.append({
        'Allocation': label,
        'Vol Weight': f'{row["w_vol"]:.0%}',
        'SPY Weight': f'{row["w_spy"]:.0%}',
        'CAGR': f'{row["cagr"]:.1%}',
        'Volatility': f'{row["vol"]:.1%}',
        'Sharpe': f'{row["sharpe"]:.2f}',
        'Sortino': f'{row["sortino"]:.2f}',
        'Max DD': f'{row["max_dd"]:.1%}',
        'Terminal ($1)': f'${row["terminal"]:.2f}',
    })

summary_df = pd.DataFrame(summary_rows).set_index('Allocation')
display(summary_df)

## Notes

1. **Pre-2008 data** excluded due to quality issues in old-format CBOE archive files.

2. **SVXY leverage change**: After Volmageddon (Feb 5, 2018), SVXY switched from −1× to −0.5×.
   This structural break is **in the test set**, so it stress-tests the strategy realistically.

3. **Transaction costs** are not included (consistent with Buehler).

4. **Settle/Close fallback**: Early 2013 CBOE files have Settle=0 with prices in Close;
   the loader falls back to Close automatically.

5. **Daily rebalancing** for the portfolio blend is assumed (constant-weight).
   Monthly rebalancing would be more practical and give similar results.